# Neural Machine Translation for India
## Course Project: CS779

#### Name: Dhruv Gupta
#### Roll No: 240354
#### Email: dhruvgupta24@iitk.ac.in

## A brief about the model architecture:

After trying both Seq2Seq models and transformers as detailed in the report, the final model selected was a transformer pipeline with the following key features:

- Transformer Architecture built from scratch
- **Weight Trying**: The decoder embedding and final output layer have the same weights in the model. (described in report)
- **Custom Learning Rate Scheduler**: Implementation of the 'Noam' optimizer (from the Attention is all you need paper), ie warmup and then inverse square root decay of the learning rate.

# 1. Imports and Setup

Imports: (uses nltk for tokenization, pytorch for all the implementation)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import json
import numpy as np
import pandas as pd
import re
import string
import nltk
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import time
import math
import os
import zipfile

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

initial config and defining global constants


In [2]:
# special tokens
PAD_token = 0
SOS_token = 1
EOS_token = 2

# use gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# file paths
TRAIN_FILE_PATH = "/kaggle/input/testdata-for-nmt/train_data1.json"
VAL_FILE_PATH = "/kaggle/input/testdata-for-nmt/val_data1.json"
TEST_FILE_PATH = "/kaggle/input/testdata-for-nmt/test_data1.json"

# training for 21 epochs.
NUM_EPOCHS = 21

Using device: cuda


# 2. Dataloaders

In [3]:
# load train
try:
    with open(TRAIN_FILE_PATH, 'r') as f:
        train_data = json.load(f)
    print("training data loaded sucess!")
except FileNotFoundError:
    print("training data not found, recheck path")
    
# load val
try:
    with open(VAL_FILE_PATH, 'r') as f:
        val_data = json.load(f)
    print("validation data loaded sucess!")
except FileNotFoundError:
    print("validation data not found, recheck path")

# load test
try:
    with open(TEST_FILE_PATH, 'r') as f:
        test_data = json.load(f)
    print("test data loaded sucess!")
except FileNotFoundError:
    print("test data not found, recheck path")

training data loaded sucess!
validation data loaded sucess!
test data loaded sucess!


# 3. Vocabulary Building and Preprocessing

In [4]:
def extract_data(data, lang_pair_key, split):
    """Extract the source and target sentences from the json files loaded"""
    # note: here split can be "Test", "Train", or "Validation"
    source_sentences = []
    target_sentences = []
    entries = []
    
    # check for existence of lang_pair_key
    if data is None or lang_pair_key not in data:
        print(f"no data found for lang pair: {lang_pair_key} in {split} data.")
        return source_sentences, target_sentences, entries
    if split not in data[lang_pair_key]:
        print(f"no split '{split}' found for lang pair: {lang_pair_key}.")
        return source_sentences, target_sentences, entries
    
    data_split = data[lang_pair_key][split]
        
    for entry_id, entry_data in data_split.items():
        entries.append(entry_id)
        source_sentences.append(entry_data["source"])

        # test and val sets wont have 'target' key
        if "target" in entry_data:
            target_sentences.append(entry_data["target"])
        
    if not target_sentences and split == "Train":
        print(f"Warning: No target sentences found for {split} split.")
        return source_sentences, None, entries
        
    if split != "Train":
        return source_sentences, None, entries

    return source_sentences, target_sentences, entries


def preprocess_sentence(sentence, language = 'en'):
    """preprocessing pipeline: lowercase -> normalise -> tokenize"""
    sentence = sentence.lower()
    sentence = re.sub(r'[' + string.punctuation + ']', '', sentence)
    sentence = re.sub(r'[0-9]+', '', sentence)
    if language == 'tgt':
        sentence = re.sub(r'[a-zA-Z]', '', sentence)
    sentence = re.sub(r'\\s+', ' ', sentence).strip()
    return nltk.word_tokenize(sentence)

In [5]:
# build vocabulary 
class Vocab:
    def __init__(self):
        # build a word to index map
        self.word2index = {"<PAD>": PAD_token, "<SOS>": SOS_token, "<EOS>": EOS_token}
        self.index2word = {PAD_token: "<PAD>", SOS_token: "<SOS>", EOS_token: "<EOS>"}
        self.number_of_words = 3 # PAD, SOS, EOS
    
    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.number_of_words
            self.index2word[self.number_of_words] = word
            self.number_of_words += 1

    def add_sentence(self, sentence):
        for word in sentence:
            self.add_word(word)

Note: setting max length to 25 for all sequences because according to the EDA done by me and as reasoned in the report.

In [6]:
MAX_LENGTH = 25 # setting this as default as well, most likely not going to need this according to EDA done

def padding_encoding_func(sentence_tokenized, vocab, max_length = 25):
    """pads or truncates the tokenised sentence sequence according to max length"""
    # start with start of sentence token
    encoded_sentence = [vocab.word2index["<SOS>"]]
    for token in sentence_tokenized:
        if token in vocab.word2index:
            encoded_sentence.append(vocab.word2index[token])
    # end with end of sentence token
    encoded_sentence.append(vocab.word2index["<EOS>"])
    # truncate if length > max
    if len(encoded_sentence) > max_length:
        encoded_sentence = encoded_sentence[:max_length - 1] + [vocab.word2index["<EOS>"]]
    # pad if length < max
    while len(encoded_sentence) < max_length:
        encoded_sentence.insert(-1, vocab.word2index["<PAD>"])
    return encoded_sentence

def time_epochs(start_time, end_time):
    """just a helper function to help me gauge the time per epoch"""
    passed_time = end_time - start_time
    passed_mins = int(passed_time / 60)
    rem_secs = int(passed_time - (passed_mins * 60))
    return passed_mins, rem_secs

# 3. Model Architecture 

Transformer + Weight Trying model

In [ ]:
class PositionalEncoding(nn.Module):
    """adds positional info to the input embeddings"""
    def __init__(self, d_model, dropout_p, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout_p)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0).transpose(0, 1) # shape: (max_len, 1, d_model)
        self.register_buffer('pe', pe) # register as buffer so it's not a model parameter

    def forward(self, x):
        # x shape: (seq_len, batch_size, d_model)
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention mechanism from scratch."""
    def __init__(self, d_model, n_heads, dropout_p):
        super(MultiHeadAttention, self).__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads # dim of each head
        
        # make q, k, v matrices
        self.fc_q = nn.Linear(d_model, d_model)
        self.fc_k = nn.Linear(d_model, d_model)
        self.fc_v = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(p=dropout_p)
        self.scale = torch.sqrt(torch.FloatTensor([self.d_k])).to(device)
        
    def forward(self, query, key, value, mask=None):
        # query, key, value shape: (batch_size, seq_len, d_model)
        batch_size = query.shape[0]
        
        # pass through linear layers
        Q = self.fc_q(query)
        K = self.fc_k(key)
        V = self.fc_v(value)
        
        # 2. rehape for multi-head attention
        # shape -> (batch_size, n_heads, seq_len, d_k)
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 1, 3)
        
        # 3. scaled dot-product attention
        # Q shape: (batch_size, n_heads, seq_len, d_k)
        # K.permute(...) shape: (batch_size, n_heads, d_k, seq_len)
        # energy shape: (batch_size, n_heads, seq_len, seq_len)
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        
        if mask is not None:
            # mask is large negative value where we want to mask
            energy = energy.masked_fill(mask == 0, -1e10)
            
        attention_weights = torch.softmax(energy, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # x shape: (batch_size, n_heads, seq_len, d_k)
        x = torch.matmul(attention_weights, V)
        
        # concat heads and send throguh final linear layer
        # x shape: (batch_size, seq_len, n_heads, d_k)
        x = x.permute(0, 2, 1, 3).contiguous() 
        # x shape: (batch_size, seq_len, d_model)
        x = x.view(batch_size, -1, self.d_model)
        x = self.fc_out(x)
        
        return x, attention_weights

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout_p):
        super(PositionwiseFeedForward, self).__init__()
        self.fc_1 = nn.Linear(d_model, d_ff)
        self.fc_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(p=dropout_p)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = self.dropout(torch.relu(self.fc_1(x)))
        x = self.fc_2(x)
        return x

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout_p):
        super(TransformerEncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout_p)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout_p)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(p=dropout_p)
        self.dropout2 = nn.Dropout(p=dropout_p)
        
    def forward(self, src, src_mask):
        # src shape: (batch_size, src_seq_len, d_model)
        # src_mask shape: (batch_size, 1, 1, src_seq_len)
        
        # self-attention (with residual connection and norm)
        _src, _ = self.self_attn(src, src, src, src_mask)
        src = self.norm1(src + self.dropout1(_src))
        
        # feed forward (with residual connection and norm)
        _src = self.ffn(src)
        src = self.norm2(src + self.dropout2(_src))
        
        return src

class TransformerEncoder(nn.Module):
    def __init__(self, input_dim, d_model, n_heads, d_ff, n_layers, dropout_p, max_len=100):
        super(TransformerEncoder, self).__init__()
        self.tok_embedding = nn.Embedding(input_dim, d_model, padding_idx=PAD_token)
        self.pos_embedding = PositionalEncoding(d_model, dropout_p, max_len)
        self.layers = nn.ModuleList([TransformerEncoderLayer(d_model, n_heads, d_ff, dropout_p) 
                                     for _ in range(n_layers)])
        self.dropout = nn.Dropout(p=dropout_p)
        self.scale = torch.sqrt(torch.FloatTensor([d_model])).to(device)
        
    def forward(self, src, src_mask):
        # src shape: (batch_size, src_seq_len)
        # src_mask shape: (batch_size, 1, 1, src_seq_len)
        batch_size = src.shape[0]
        src_seq_len = src.shape[1]
        
        # src shape: (batch_size, src_seq_len, d_model)
        src_emb = self.tok_embedding(src) * self.scale
        
        # Transpose for positional encoding (seq_len, batch_size, d_model)
        src_pos = self.pos_embedding(src_emb.transpose(0, 1))
        
        # Transpose back to (batch_size, seq_len, d_model)
        src = self.dropout(src_pos.transpose(0, 1))
        
        for layer in self.layers:
            src = layer(src, src_mask)
            
        return src

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout_p):
        super(TransformerDecoderLayer, self).__init__()
        self.masked_self_attn = MultiHeadAttention(d_model, n_heads, dropout_p)
        self.encoder_attn = MultiHeadAttention(d_model, n_heads, dropout_p)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout_p)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(p=dropout_p)
        self.dropout2 = nn.Dropout(p=dropout_p)
        self.dropout3 = nn.Dropout(p=dropout_p)
        
    def forward(self, tgt, enc_src, tgt_mask, src_mask):
        # tgt shape: (batch_size, tgt_seq_len, d_model)
        # enc_src shape: (batch_size, src_seq_len, d_model)
        # tgt_mask shape: (batch_size, 1, tgt_seq_len, tgt_seq_len)
        # src_mask shape: (batch_size, 1, 1, src_seq_len)
        
        # masked self-attention (with residual and norm)
        _tgt, _ = self.masked_self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = self.norm1(tgt + self.dropout1(_tgt))

        # encoder-decoder attention (with residual and norm)
        # Query=tgt, Key=enc_src, Value=enc_src
        _tgt, attention = self.encoder_attn(tgt, enc_src, enc_src, src_mask)
        tgt = self.norm2(tgt + self.dropout2(_tgt))

        # feed forward (with residual and norm)
        _tgt = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout3(_tgt))
        
        return tgt, attention

class TransformerDecoder(nn.Module):
    def __init__(self, output_dim, d_model, n_heads, d_ff, n_layers, dropout_p, max_len=100):
        super(TransformerDecoder, self).__init__()
        self.tok_embedding = nn.Embedding(output_dim, d_model, padding_idx=PAD_token)
        self.pos_embedding = PositionalEncoding(d_model, dropout_p, max_len)
        self.layers = nn.ModuleList([TransformerDecoderLayer(d_model, n_heads, d_ff, dropout_p)
                                     for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, output_dim)
        self.dropout = nn.Dropout(p=dropout_p)
        self.scale = torch.sqrt(torch.FloatTensor([d_model])).to(device)
        
    def forward(self, tgt, enc_src, tgt_mask, src_mask):
        # tgt shape: (batch_size, tgt_seq_len)
        # enc_src shape: (batch_size, src_seq_len, d_model)
        # tgt_mask shape: (batch_size, 1, tgt_seq_len, tgt_seq_len)
        # src_mask shape: (batch_size, 1, 1, src_seq_len)
        
        # tgt shape: (batch_size, tgt_seq_len, d_model)
        tgt_emb = self.tok_embedding(tgt) * self.scale
        
        # transpsoe for positional encoding (tgt_seq_len, batch_size, d_model)
        tgt_pos = self.pos_embedding(tgt_emb.transpose(0, 1))
        
        # transpose back to (batch_size, tgt_seq_len, d_model)
        tgt = self.dropout(tgt_pos.transpose(0, 1))
        
        for layer in self.layers:
            tgt, attention = layer(tgt, enc_src, tgt_mask, src_mask)
            
        # output shape: (batch_size, tgt_seq_len, output_dim)
        output = self.fc_out(tgt)
        
        return output, attention

class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, tgt_pad_idx, device):
        super(Transformer, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx
        self.device = device
        
        # WEIGHT TRYING IMPLEMENTATION
        # make weights same for decoder embedding layer and final linear layer
        self.decoder.fc_out.weight = self.decoder.tok_embedding.weight
        print("Applied Weight Tying between Decoder Embedding and Final FC Layer.")
        
    def make_src_mask(self, src):
        # src shape: (batch_size, src_seq_len)
        # src_mask shape: (batch_size, 1, 1, src_seq_len)
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask
    
    def make_tgt_mask(self, tgt):
        # tgt shape: (batch_size, tgt_seq_len)
        
        # target pad mask
        # tgt_pad_mask shape: (batch_size, 1, 1, tgt_seq_len)
        tgt_pad_mask = (tgt != self.tgt_pad_idx).unsqueeze(1).unsqueeze(2)
        
        # target look ahead mask (subsequent mask)
        tgt_seq_len = tgt.shape[1]
        # tgt_sub_mask shape: (tgt_seq_len, tgt_seq_len)
        tgt_sub_mask = torch.tril(torch.ones((tgt_seq_len, tgt_seq_len), device=self.device)).bool()
        
        # combine both masks
        # tgt_mask shape: (batch_size, 1, tgt_seq_len, tgt_seq_len)
        tgt_mask = tgt_pad_mask & tgt_sub_mask
        return tgt_mask

    def forward(self, src, tgt):
        # src shape: (batch_size, src_seq_len)
        # tgt shape: (batch_size, tgt_seq_len)
        
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        
        # enc_src shape: (batch_size, src_seq_len, d_model)
        enc_src = self.encoder(src, src_mask)
        
        # output shape: (batch_size, tgt_seq_len, output_dim)
        output, attention = self.decoder(tgt, enc_src, tgt_mask, src_mask)
        
        return output

# 4. custom LR scheduler (Noam optimizer)

In [8]:
class NoamOpt:
    """optimizer wrapper func that implements the transformer leraning rate schedule from attention is all you need paper"""
    def __init__(self, d_model, factor, warmup, optimizer):
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.d_model = d_model
        self._rate = 0
        
    def step(self):
        "Update parameters and rate"
        self._step += 1
        rate = self.rate()
        for p in self.optimizer.param_groups:
            p['lr'] = rate
        self._rate = rate
        self.optimizer.step()
        
    def rate(self, step = None):
        "Implement the lrate formula"
        if step is None:
            step = self._step
        return self.factor * \
            (self.d_model ** (-0.5) *
            min(step ** (-0.5), step * self.warmup ** (-1.5)))
    
    def zero_grad(self):
        self.optimizer.zero_grad()
        
def get_std_opt(model, d_model, warmup, factor=1):
    "Helper function to create the Noam optimizer"
    # The paper used lr=0, betas=(0.9, 0.98), eps=1e-9
    return NoamOpt(d_model, factor, warmup,
            torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))

# RUNNING FOR ENGLISH TO BENGALI

loading data:

In [9]:
LANG_PAIR_B = "English-Bengali"  
OUTPUT_CSV_NAME_B = "answersB.csv"
MODEL_NAME_B = "model-bengali-transformer.pt"

# extract bengali data using above func
print("starting bengali data extraction...")
source_sentences_train_b, target_sentences_train_b, train_ids_b = extract_data(train_data, LANG_PAIR_B, "Train")
source_sentences_test_b, _, test_ids_b = extract_data(test_data, LANG_PAIR_B, "Test")

# bengali data summary
print(f"Summary: Train examples - {len(source_sentences_train_b)}, Test examples - {len(source_sentences_test_b)}")

# preprocess bengali
print("starting bengali data preprocessing...")
en_train_tokens_b = [preprocess_sentence(sentence, language='en') for sentence in tqdm(source_sentences_train_b)]
target_train_tokens_b = [preprocess_sentence(sentence, language='bn') for sentence in tqdm(target_sentences_train_b)]   
en_test_tokens_b = [preprocess_sentence(sentence, language='en') for sentence in tqdm(source_sentences_test_b)]


starting bengali data extraction...
Summary: Train examples - 68849, Test examples - 19672
starting bengali data preprocessing...


100%|██████████| 19672/19672 [00:01<00:00, 12070.82it/s]


building vocabulary:

In [10]:
en_vocab_b = Vocab()
target_vocab_b = Vocab()

print("building english vocab for bengali model..")
for sentence_tokens in tqdm(en_train_tokens_b):
    en_vocab_b.add_sentence(sentence_tokens)
print("building bengali vocab for bengali model..")
for sentence_tokens in tqdm(target_train_tokens_b):
    target_vocab_b.add_sentence(sentence_tokens)

print(f"Bengali vocab: Source (english) size: {en_vocab_b.number_of_words}, Target (bengali) size: {target_vocab_b.number_of_words}")


building english vocab for bengali model..


100%|██████████| 68849/68849 [00:00<00:00, 357260.91it/s]


building bengali vocab for bengali model..


100%|██████████| 68849/68849 [00:00<00:00, 287049.33it/s]

Bengali vocab: Source (english) size: 53923, Target (bengali) size: 105528


encoding and padding:

In [11]:
print("encoding and padding bengali data...")
en_train_encoded_b = [padding_encoding_func(sentence, en_vocab_b, MAX_LENGTH) for sentence in tqdm(en_train_tokens_b)]
target_train_encoded_b = [padding_encoding_func(sentence, target_vocab_b, MAX_LENGTH) for sentence in tqdm(target_train_tokens_b)]
en_test_encoded_b = [padding_encoding_func(sentence, en_vocab_b, MAX_LENGTH) for sentence in tqdm(en_test_tokens_b)]

BATCH_SIZE = 64

# create tensors
en_train_tensor_b = torch.LongTensor(en_train_encoded_b).to(device)
target_train_tensor_b = torch.LongTensor(target_train_encoded_b).to(device)
en_test_tensor_b = torch.LongTensor(en_test_encoded_b).to(device)

# create dataset + dataloaders
train_dataset_b = TensorDataset(en_train_tensor_b, target_train_tensor_b)
train_dataloader_b = DataLoader(train_dataset_b, batch_size=BATCH_SIZE, shuffle=True)
test_dataset_b = TensorDataset(en_test_tensor_b)
test_dataloader_b = DataLoader(test_dataset_b, batch_size=1, shuffle=False) # BATCH SIZE IS 1 HERE, WHY? :)

print(f"bengali train dataloader: {len(train_dataloader_b)} batches of size {BATCH_SIZE}")
print(f"bengali test dataloader: {len(test_dataloader_b)} batches of size 1")

encoding and padding bengali data...


100%|██████████| 19672/19672 [00:00<00:00, 215671.04it/s]


bengali train dataloader: 1076 batches of size 64
bengali test dataloader: 19672 batches of size 1


training the model for 21 epochs using noam opt and weight trying

In [12]:
def train_func(model, dataloader, optimizer, criterion, clip):
    model.train()
    current_loss = 0
    
    for batch in tqdm(dataloader, desc = "Training"):
        source, target = batch
        source = source.to(device); target = target.to(device)
        optimizer.zero_grad()
        
        # decoder input: target sequence shifted right (except for eos)
        # target for loss calculation: target sequence shifted left (except for <sos>)
        target_input = target[:, :-1]
        target_output = target[:, 1:]
        
        # forward pass
        output = model(source, target_input) # shape: (batch_size, target sequence length - 1, output_dim)
        
        # loss calculation
        # first reshape to 2d tensor
        output_dim = output.shape[-1]
        output_forlosscalc = output.reshape(-1, output_dim)
        target_forlosscalc = target_output.reshape(-1)
        loss = criterion(output_forlosscalc, target_forlosscalc)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = clip)

        optimizer.step()
        current_loss += loss.item()
    return current_loss / len(dataloader)

hyperparams and making the model

In [13]:
# hyperparams
INPUT_DIM_B = en_vocab_b.number_of_words
OUTPUT_DIM_B = target_vocab_b.number_of_words
D_MODEL = 256 # embedding dim
N_HEADS = 8 # no of attention heads 
FEEDFORWARD_DIM = 512 # feedforward layer dim
N_LAYERS = 3 # no of encoder/decoder layers
DROPOUT = 0.1
CLIP = 1.0
WARMUP_STEPS_B = 4000 # usually 4k is warmup, might try to increase later

encoder_b = TransformerEncoder(INPUT_DIM_B, D_MODEL, N_HEADS, FEEDFORWARD_DIM, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)
decoder_b = TransformerDecoder(OUTPUT_DIM_B, D_MODEL, N_HEADS, FEEDFORWARD_DIM, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)

model_b = Transformer(encoder_b, decoder_b, PAD_token, PAD_token, device).to(device)

# init weights
def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name and param.dim() >= 2:
            nn.init.xavier_uniform_(param.data)
        elif 'bias' in name:
            nn.init.constant_(param.data, 0)
model_b.apply(init_weights)

# noam optimizer
optimizer_b = get_std_opt(model_b, D_MODEL, WARMUP_STEPS_B)
criterion_b = nn.CrossEntropyLoss(ignore_index=PAD_token)

# summary 
print("bengali model summary:")
print(f"source (english) vocab size: {INPUT_DIM_B}")
print(f"Target (bengali) vocab size: {OUTPUT_DIM_B}")
print(f"model embedding dim: {D_MODEL}, number of heads: {N_HEADS}, number of encoder/decoder layrs: {N_LAYERS}")


Applied Weight Tying between Decoder Embedding and Final FC Layer.
bengali model summary:
source (english) vocab size: 53923
Target (bengali) vocab size: 105528
model embedding dim: 256, number of heads: 8, number of encoder/decoder layrs: 3


hyperparams and making the model

training loop:

In [14]:
print("starting bengali training...")

training_losses_b = []
for epoch in range(NUM_EPOCHS):
    start_timestamp = time.time()
    training_loss = train_func(model_b, train_dataloader_b, optimizer_b, criterion_b, CLIP)
    training_losses_b.append(training_loss)
    end_timestamp = time.time()
    epoch_mins, epoch_secs = time_epochs(start_timestamp, end_timestamp)
    epoch_num = epoch + 1
    print(f'epoch: {epoch_num:02} | time: {epoch_mins}m {epoch_secs}s | training loss: {training_loss:.3f}')
    
print("training finished, saving bengali model..")
torch.save(model_b.state_dict(), MODEL_NAME_B)
print(f"saved final bengali model as {MODEL_NAME_B}")


starting bengali training...


Training: 100%|██████████| 1076/1076 [01:22<00:00, 12.98it/s]


epoch: 01 | time: 1m 22s | training loss: 8.999


Training: 100%|██████████| 1076/1076 [01:22<00:00, 13.11it/s]


epoch: 02 | time: 1m 22s | training loss: 7.518


...

epoch: 19 | time: 1m 22s | training loss: 4.221


Training: 100%|██████████| 1076/1076 [01:22<00:00, 13.10it/s]


epoch: 20 | time: 1m 22s | training loss: 4.170


Training: 100%|██████████| 1076/1076 [01:22<00:00, 13.11it/s]


epoch: 21 | time: 1m 22s | training loss: 4.125
training finished, saving bengali model..
saved final bengali model as model-bengali-transformer.pt


inference for bengali:

In [15]:
def get_translations(tokenised_sentence, english_vocab, target_vocab, model, device, max_length = MAX_LENGTH):
    """performs inference and returns the translated sentence when given a tokenised sentence as input"""
    model.eval()
    
    # encode, pad, create tensor
    encoded_sentence = padding_encoding_func(tokenised_sentence, english_vocab, max_length)
    source_tensor = torch.LongTensor(encoded_sentence).unsqueeze(0).to(device)
    
    # source mask
    source_mask = model.make_src_mask(source_tensor)
    
    # run encoder one time
    with torch.no_grad():
        encoder_source = model.encoder(source_tensor, source_mask)
    
    # start decoder input with <sos> token first
    # target indices shape: (1 (since batch size 1), sequence length is 1 initially since only <sos>)
    target_indices = [SOS_token]
    
    for i in range(max_length):
        target_tensor = torch.LongTensor(target_indices).unsqueeze(0).to(device)
        
        # target mask
        target_mask = model.make_tgt_mask(target_tensor)
        
        with torch.no_grad():
            output, attention = model.decoder(target_tensor, encoder_source, target_mask, source_mask)
        
        # get the next token prediction from the output
        pred_token = output.argmax(2)[:, -1].item()
        target_indices.append(pred_token)
        
        # stop if <eos> is predicted
        if pred_token == EOS_token:
            break
        
    # convert indices to words using target vocab
    translated_sentence = []
    for index in target_indices[1:]: # skip <sos>
        if index == EOS_token:
            break
        translated_sentence.append(target_vocab.index2word[index])
    return " ".join(translated_sentence)


# run the inference

print("starting bengali inference on test set..")
model_path = MODEL_NAME_B
print(f"loading model: {model_path}")
try:
    model_b.load_state_dict(torch.load(model_path))
    print("model loaded sucessfully")
except FileNotFoundError:
    print("ERROR::model file not found!!")
    raise
    
    
# generate translations and save
test_translations_b = []

print(f"generating {len(en_test_tokens_b)} translations for bengali test set...")
for tokens in tqdm(en_test_tokens_b):
    translation = get_translations(tokens, en_vocab_b, target_vocab_b, model_b, device, MAX_LENGTH)
    test_translations_b.append(translation)
    
# generate csv
df_translations_b = pd.DataFrame()
df_translations_b['ID'] = test_ids_b
df_translations_b['Translation'] = test_translations_b
df_translations_b.to_csv(OUTPUT_CSV_NAME_B, index=False)
print(f"Test submission file saved as {OUTPUT_CSV_NAME_B}")

print("bengali inference complete.")

# free memory before starting hindi cos i was facing memory issues
del model_b, encoder_b, decoder_b, optimizer_b
del en_train_tensor_b, target_train_tensor_b, train_dataset_b, train_dataloader_b
del en_train_encoded_b, target_train_encoded_b
torch.cuda.empty_cache()

    

starting bengali inference on test set..
loading model: model-bengali-transformer.pt
model loaded sucessfully
generating 19672 translations for bengali test set...


100%|██████████| 19672/19672 [32:10<00:00, 10.19it/s]


Test submission file saved as answersB.csv
bengali inference complete.


# RUNNING FOR ENGLISH TO HINDI

In [16]:
LANG_PAIR_H = "English-Hindi"  
OUTPUT_CSV_NAME_H = "answersH.csv"
MODEL_NAME_H = "model-hindi-transformer.pt"

# extract hindi data using above func
print("starting hindi data extraction...")
source_sentences_train_h, target_sentences_train_h, train_ids_h = extract_data(train_data, LANG_PAIR_H, "Train")
source_sentences_test_h, _, test_ids_h = extract_data(test_data, LANG_PAIR_H, "Test")

# hindi data summary
print(f"Summary: Train examples - {len(source_sentences_train_h)}, Test examples - {len(source_sentences_test_h)}")

# preprocess hindi
print("starting hindi data preprocessing...")
en_train_tokens_h = [preprocess_sentence(sentence, language='en') for sentence in tqdm(source_sentences_train_h)]
target_train_tokens_h = [preprocess_sentence(sentence, language='hi') for sentence in tqdm(target_sentences_train_h)]   
en_test_tokens_h = [preprocess_sentence(sentence, language='en') for sentence in tqdm(source_sentences_test_h)]


starting hindi data extraction...
Summary: Train examples - 80797, Test examples - 23085
starting hindi data preprocessing...


100%|██████████| 23085/23085 [00:01<00:00, 11907.24it/s]


building vocabulary:

In [17]:
en_vocab_h = Vocab()
target_vocab_h = Vocab()

print("building english vocab for hindi model..")
for sentence_tokens in tqdm(en_train_tokens_h):
    en_vocab_h.add_sentence(sentence_tokens)
print("building hindi vocab for hindi model..")
for sentence_tokens in tqdm(target_train_tokens_h):
    target_vocab_h.add_sentence(sentence_tokens)

print(f"Hindi vocab: Source (english) size: {en_vocab_h.number_of_words}, Target (hindi) size: {target_vocab_h.number_of_words}")


building english vocab for hindi model..


100%|██████████| 80797/80797 [00:00<00:00, 345101.07it/s]


building hindi vocab for hindi model..


100%|██████████| 80797/80797 [00:00<00:00, 273117.71it/s]

Hindi vocab: Source (english) size: 57278, Target (hindi) size: 75562


encoding and padding:

In [18]:
print("encoding and padding hindi data...")
en_train_encoded_h = [padding_encoding_func(sentence, en_vocab_h, MAX_LENGTH) for sentence in tqdm(en_train_tokens_h)]
target_train_encoded_h = [padding_encoding_func(sentence, target_vocab_h, MAX_LENGTH) for sentence in tqdm(target_train_tokens_h)]
en_test_encoded_h = [padding_encoding_func(sentence, en_vocab_h, MAX_LENGTH) for sentence in tqdm(en_test_tokens_h)]

BATCH_SIZE = 64

# create tensors
en_train_tensor_h = torch.LongTensor(en_train_encoded_h).to(device)
target_train_tensor_h = torch.LongTensor(target_train_encoded_h).to(device)
en_test_tensor_h = torch.LongTensor(en_test_encoded_h).to(device)

# create dataset + dataloaders
train_dataset_h = TensorDataset(en_train_tensor_h, target_train_tensor_h)
train_dataloader_h = DataLoader(train_dataset_h, batch_size=BATCH_SIZE, shuffle=True)
test_dataset_h = TensorDataset(en_test_tensor_h)
test_dataloader_h = DataLoader(test_dataset_h, batch_size=1, shuffle=False) # BATCH SIZE IS 1 HERE, WHY? :)

print(f"hindi train dataloader: {len(train_dataloader_h)} batches of size {BATCH_SIZE}")
print(f"hindi test dataloader: {len(test_dataloader_h)} batches of size 1")

encoding and padding hindi data...


100%|██████████| 23085/23085 [00:00<00:00, 211000.76it/s]


hindi train dataloader: 1263 batches of size 64
hindi test dataloader: 23085 batches of size 1


training the model for 21 epochs using noam opt and weight trying

In [19]:
def train_func(model, dataloader, optimizer, criterion, clip):
    model.train()
    current_loss = 0
    
    for batch in tqdm(dataloader, desc = "Training"):
        source, target = batch
        source = source.to(device); target = target.to(device)
        optimizer.zero_grad()
        
        # decoder input: target sequence shifted right (except for eos)
        # target for loss calculation: target sequence shifted left (except for <sos>)
        target_input = target[:, :-1]
        target_output = target[:, 1:]
        
        # forward pass
        output = model(source, target_input) # shape: (batch_size, target sequence length - 1, output_dim)
        
        # loss calculation
        # first reshape to 2d tensor
        output_dim = output.shape[-1]
        output_forlosscalc = output.reshape(-1, output_dim)
        target_forlosscalc = target_output.reshape(-1)
        loss = criterion(output_forlosscalc, target_forlosscalc)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = clip)

        optimizer.step()
        current_loss += loss.item()
    return current_loss / len(dataloader)



hyperparams and making the model

In [20]:
# hyperparams
INPUT_DIM_H = en_vocab_h.number_of_words
OUTPUT_DIM_H = target_vocab_h.number_of_words
D_MODEL = 256 # embedding dim
N_HEADS = 8 # no of attention heads 
FEEDFORWARD_DIM = 512 # feedforward layer dim
N_LAYERS = 3 # no of encoder/decoder layers
DROPOUT = 0.1
CLIP = 1.0
WARMUP_STEPS_H = 4000 # usually 4k is warmup, might try to increase later

encoder_h = TransformerEncoder(INPUT_DIM_H, D_MODEL, N_HEADS, FEEDFORWARD_DIM, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)
decoder_h = TransformerDecoder(OUTPUT_DIM_H, D_MODEL, N_HEADS, FEEDFORWARD_DIM, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)

model_h = Transformer(encoder_h, decoder_h, PAD_token, PAD_token, device).to(device)

# init weights
def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name and param.dim() >= 2:
            nn.init.xavier_uniform_(param.data)
        elif 'bias' in name:
            nn.init.constant_(param.data, 0)
model_h.apply(init_weights)

# noam optimizer
optimizer_h = get_std_opt(model_h, D_MODEL, WARMUP_STEPS_H)
criterion_h = nn.CrossEntropyLoss(ignore_index=PAD_token)

# summary
print("hindi model summary:")
print(f"source (english) vocab size: {INPUT_DIM_H}")
print(f"Target (hindi) vocab size: {OUTPUT_DIM_H}")
print(f"model embedding dim: {D_MODEL}, number of heads: {N_HEADS}, number of encoder/decoder layrs: {N_LAYERS}")


Applied Weight Tying between Decoder Embedding and Final FC Layer.
hindi model summary:
source (english) vocab size: 57278
Target (hindi) vocab size: 75562
model embedding dim: 256, number of heads: 8, number of encoder/decoder layrs: 3


training loop:

In [21]:
print("starting hindi training...")

training_losses_h = []
for epoch in range(NUM_EPOCHS):
    start_timestamp = time.time()
    training_loss = train_func(model_h, train_dataloader_h, optimizer_h, criterion_h, CLIP)
    training_losses_h.append(training_loss)
    end_timestamp = time.time()
    epoch_mins, epoch_secs = time_epochs(start_timestamp, end_timestamp)
    epoch_num = epoch + 1
    print(f'epoch: {epoch_num:02} | time: {epoch_mins}m {epoch_secs}s | training loss: {training_loss:.3f}')

print("training finished, saving hindi model..")
torch.save(model_h.state_dict(), MODEL_NAME_H)
print(f"saved final hindi model as {MODEL_NAME_H}")


starting hindi training...


Training: 100%|██████████| 1263/1263 [01:21<00:00, 15.51it/s]


epoch: 01 | time: 1m 21s | training loss: 7.678


Training: 100%|██████████| 1263/1263 [01:21<00:00, 15.54it/s]


epoch: 02 | time: 1m 21s | training loss: 5.851


...

epoch: 19 | time: 1m 21s | training loss: 2.668


Training: 100%|██████████| 1263/1263 [01:21<00:00, 15.53it/s]


epoch: 20 | time: 1m 21s | training loss: 2.635


Training: 100%|██████████| 1263/1263 [01:21<00:00, 15.53it/s]


epoch: 21 | time: 1m 21s | training loss: 2.602
training finished, saving hindi model..
saved final hindi model as model-hindi-transformer.pt


inference for hindi:

In [22]:
def get_translations(tokenised_sentence, english_vocab, target_vocab, model, device, max_length = MAX_LENGTH):
    """performs inference and returns the translated sentence when given a tokenised sentence as input"""
    model.eval()
    
    # encode, pad, create tensor
    encoded_sentence = padding_encoding_func(tokenised_sentence, english_vocab, max_length)
    source_tensor = torch.LongTensor(encoded_sentence).unsqueeze(0).to(device)
    
    # source mask
    source_mask = model.make_src_mask(source_tensor)
    
    # run encoder one time
    with torch.no_grad():
        encoder_source = model.encoder(source_tensor, source_mask)
    
    # start decoder input with <sos> token first
    # target indices shape: (1 (since batch size 1), sequence length is 1 initially since only <sos>)
    target_indices = [SOS_token]
    
    for i in range(max_length):
        target_tensor = torch.LongTensor(target_indices).unsqueeze(0).to(device)
        
        # target mask
        target_mask = model.make_tgt_mask(target_tensor)
        
        with torch.no_grad():
            output, attention = model.decoder(target_tensor, encoder_source, target_mask, source_mask)
        
        # get the next token prediction from the output
        pred_token = output.argmax(2)[:, -1].item()
        target_indices.append(pred_token)
        
        # stop if <eos> is predicted
        if pred_token == EOS_token:
            break
        
    # convert indices to words using target vocab
    translated_sentence = []
    for index in target_indices[1:]: # skip <sos>
        if index == EOS_token:
            break
        translated_sentence.append(target_vocab.index2word[index])
    return " ".join(translated_sentence)


# run the inference

print("starting hindi inference on test set..")
model_path = MODEL_NAME_H
print(f"loading model: {model_path}")
try:
    model_h.load_state_dict(torch.load(model_path))
    print("model loaded sucessfully")
except FileNotFoundError:
    print("ERROR::model file not found!!")
    raise
    
    
# generate translations and save
test_translations_h = []

print(f"generating {len(en_test_tokens_h)} translations for hindi test set...")
for tokens in tqdm(en_test_tokens_h):
    translation = get_translations(tokens, en_vocab_h, target_vocab_h, model_h, device, MAX_LENGTH)
    test_translations_h.append(translation)

# generate csv
df_translations_h = pd.DataFrame()
df_translations_h['ID'] = test_ids_h
df_translations_h['Translation'] = test_translations_h
df_translations_h.to_csv(OUTPUT_CSV_NAME_H, index=False)
print(f"Test submission file saved as {OUTPUT_CSV_NAME_H}")

print("hindi inference complete.")

starting hindi inference on test set..
loading model: model-hindi-transformer.pt
model loaded sucessfully
generating 23085 translations for hindi test set...


100%|██████████| 23085/23085 [36:35<00:00, 10.51it/s]

Test submission file saved as answersH.csv
hindi inference complete.


# Combining, formatting and zipping for final submission

In [23]:
print("starting combination of hindi and bengali csvs")

# file paths
bengali_csv_path = "answersB.csv"
hindi_csv_path = "answersH.csv"
combined_csv_path = "answersBH.csv"
final_answer_path = "answer.csv"
zip_path = "submission.zip"

# combine
try:
    df_bengali = pd.read_csv(bengali_csv_path)
    df_hindi = pd.read_csv(hindi_csv_path)
    df_combined = pd.concat([df_bengali, df_hindi])
    df_combined.to_csv(combined_csv_path, index=False) # creates answersBH.csv
    print(f"Combined {len(df_bengali)} bengali and {len(df_hindi)} hindi translations into {combined_csv_path}")
except FileNotFoundError as e:
    print(f"error: files couldnt be found!! {e}")
    raise

# format to final answer.csv file -> REPLACE NULL VALUES WITH SOME RANDOM TRANSLATION AND ENSURE EVERYTHNG IN QUOTES
try:
    final_df = pd.read_csv(combined_csv_path)
    # replace nulls with some random translation
    final_df['Translation'] = final_df['Translation'].fillna('some random translation')
    
    with open(final_answer_path, 'w', encoding='utf-8') as f:
        f.writelines("ID,Translation\n") # i think comma sepearated is meant to be used, not tab??
        for i, row in final_df.iterrows():
            translated_text = str(row["Translation"])
            if translated_text.strip() == '':
                translated_text = 'some random translation'
            translated_text = translated_text.replace('\"', '\"\"') # ensure everything in quotes
            f.writelines(f'{row["ID"]},\"{translated_text}\"\n')
    print("created final answer.csv file successfully!!")
except Exception as e:
    print(f"error: {e}")
    raise
    
# zip file
try:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(final_answer_path)
    print(f"zipped final answer.csv into {zip_path} successfully!!")
except Exception as e:
    print(f"error while zipping: {e}")
    raise

print("="*50)
print("ALL PROCESSING COMPLETE!")
print(f"Files generated: {OUTPUT_CSV_NAME_B}, {OUTPUT_CSV_NAME_H}, {combined_csv_path}, {final_answer_path}, {zip_path}")
print("="*50)

starting combination of hindi and bengali csvs
Combined 19672 bengali and 23085 hindi translations into answersBH.csv
created final answer.csv file successfully!!
zipped final answer.csv into submission.zip successfully!!
ALL PROCESSING COMPLETE!
Files generated: answersB.csv, answersH.csv, answersBH.csv, answer.csv, submission.zip
